# Traceprop-LLM — GPU evidence run: LogIX comparison, Pythia-2.8B scale, same-setting overhead

Run order (per priority — item 12 decides whether the systems contribution holds up, so it goes first):
1. **LogIX vs. LoRAGradientLogger** — same model, scope, batch/seq, GPU, seed. Two LogIX flush configs (its own disk-write default, and a matched-buffering config comparable to our `buffer=True`/`drain()`).
2. **Pythia-2.8B row for Table 1** — reuses `exp25`, now fixed to 20 repeats / 10 warmup by default (the old 5-repeat default is what produced the noisy Pythia-410M ±2.43% number already in the paper — rerun that row too while you're at it).
3. **Same-setting overhead + LDS (item 10)** — most expensive, run last, only if time/budget allows.

**CPU proxy already run (tiny synthetic classifier, matched scope, 100 repeats):** LogIX 12.9% ± 19.3%, LoRAGradientLogger 25.9% ± 14.7% — LogIX faster. Opposite regime from the paper's claim (small CPU model vs. GPT-2/Pythia on GPU), so this doesn't resolve anything on its own.

**Use the L4.** Table 1 was measured on an L4, so new numbers are directly comparable, and 24GB fits Pythia-2.8B LoRA in bf16. Don't switch to A100 unless you hit OOM — it costs more compute units and mixing GPUs weakens the tables.

## Setup: pin versions, mount Drive, log GPU

Run this once per session. Pinning versions here means a disconnect-and-reconnect mid-run reinstalls the *same* environment, not whatever the latest release happens to be that day.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Pin to known-good versions -- an unpinned `transformers` install can pull a
# broken bleeding-edge release (hit this locally: NameError in
# transformers/integrations/accelerate.py on an unpinned install). Check
# https://github.com/logix-project/logix for its current tested transformers
# range before trusting this pin.
#
# logix-ai's setup.py caps python_requires at <3.11, but Colab's current
# default is Python 3.13. The wheel is pure-Python (py3-none-any, no
# compiled extensions) and has no syntax/API that fails to parse under
# newer Python, so --ignore-requires-python is a reasonable bypass here --
# just verify the import + a trivial logix.init()/watch() call actually
# work below before trusting any timing numbers from it.
!pip -q install "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" datasets
!pip -q install --ignore-requires-python logix-ai
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
import logix; print('logix ok:', logix.__file__)

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

In [ ]:
%cd /content/Traceprop/experiments
import torch, torch.nn as nn
import logix

m = nn.Sequential(nn.Linear(16, 8), nn.Linear(8, 2)).cuda()
run = logix.LogIX(project="smoketest", config="exp31_config.yaml")
run.watch(m, name_filter=["0", "1"], type_filter=[nn.Linear])
run.setup({"grad": ["log"]})
run.save(False)

x = torch.randn(4, 16, device="cuda")
with run(data_id=["a", "b", "c", "d"]):
    m.zero_grad(set_to_none=True)
    out = m(x)
    out.sum().backward()

print("LogIX smoke test OK -- import, hook attach, and one forward/backward under `with run(...)` all worked.")

## Run 1: Traceprop's own overhead (exp25) — corrected Table 1 numbers

Now defaults to 20 repeats / 10 warmup (was 5/5 — the likely source of the Pythia-410M ±2.43% noise already in the paper). Run all three model sizes so Table 1 can be fully replaced with matched-power numbers, not just patched for one row.

In [ ]:
%cd /content/Traceprop/experiments
for model in ['gpt2', 'EleutherAI/pythia-410m', 'EleutherAI/pythia-1b']:
    !python exp25_llm_inline_overhead.py --backend hf --model {model} --device cuda \
        --steps 200 --track 1 --proj_dim 512
    !cp results/exp25_hf_*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Run 1 (continued): LogIX's overhead, same model/scope (exp31)

Now reports two configs (`logix_default_disk`, `matched_buffering`) and peak GPU memory for both baseline and LogIX. Compare `configs.matched_buffering.overhead_pct_median` against exp25's `throughput_overhead_pct` — that's the fairest apples-to-apples number (both buffered, neither paying disk I/O in the timed region).

In [ ]:
%cd /content/Traceprop/experiments
!python exp31_logix_comparison.py --backend hf --model gpt2 --device cuda \
    --steps 200 --track 1
!cp results/exp31_logix_hf_*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Result: Run 1

If Traceprop is still cheaper at GPT-2/Pythia scale (opposite of the CPU proxy), that's the real item-12 answer -- write it up as "comparable mechanism, lower overhead at the scale that matters," naming the specific implementation choices (on-device projection kept off the critical path, sparse-JL vs. LogIX's PCA/covariance-based compression) as the reason. If LogIX is still cheaper here too, that's the result that decides whether the systems contribution needs reframing around lineage integration rather than raw speed -- report it as-is, don't reach for a rationalization.

**Known gap, not yet measured here:** bytes-per-example (LogIX's rank vs. our `proj_dim` aren't yet matched to the same ~2KB/example storage budget) and LDS-on-LogIX (whether its projection gives comparable attribution quality, not just comparable speed). If Run 1's speed result is close either way, these matter for the final call and are worth a follow-up before writing the comparison into the paper.

In [ ]:
import json, glob
print('--- Traceprop (exp25), all model sizes ---')
for p in sorted(glob.glob('/content/Traceprop/experiments/results/exp25_hf_*.json')):
    d = json.load(open(p))
    print(f"  {d.get('model'):<25} throughput={d.get('throughput_overhead_pct')}% +/- {d.get('throughput_overhead_std')}  (n={d.get('repeats')})")
print('--- LogIX (exp31) ---')
for p in glob.glob('/content/Traceprop/experiments/results/exp31_logix_hf_*.json'):
    d = json.load(open(p))
    for cfg_name, cfg in d['configs'].items():
        print(f"  {d.get('model')} [{cfg_name}]  overhead={cfg['overhead_pct_median']}% +/- {cfg['overhead_pct_std']}  peak_mem(base/logix)={cfg.get('peak_mem_mb_baseline')}/{cfg.get('peak_mem_mb_logix')} MB")

## Run 2: Pythia-2.8B row for Table 1

Reuses exp25 with `--dtype bf16` (now implemented — fp32 would likely OOM a 2.8B model on a 24GB L4 once optimizer state and activations are added).

In [ ]:
%cd /content/Traceprop/experiments
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-2.8b --device cuda \
    --steps 200 --track 1 --proj_dim 512 --dtype bf16
!cp results/exp25_hf_*pythia-2.8b*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null

## Run 3 (optional, most expensive): same-setting overhead + LDS

Only run this if Runs 1-2 leave budget. Start small (GPT-2-124M config on a small corpus) and time one epoch before committing to the full run — this is the one most likely to blow the compute budget if sized wrong.

This isn't scripted yet — bring the timing from a 1-epoch trial run back to Claude Code before committing to the full run, so the epoch count / corpus size can be sized to fit your remaining compute units.